# Chapter 00-04 · Prediction, explanation, and cause

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - the ideas are simple, the consequences are not

**Prerequisites:** 00-01 to 00-03. You should be able to name a baseline, say why a
training-set score is not evidence, and classify a task as supervised or unsupervised.

**Position in the learning path:** module 00 (Orientation), chapter 4 of 4 - the last one.
Before this: **00-03**. After this: **02-01**, where module 02 starts on reading data
sceptically. (Module 01 is the optional Python bridge; take its diagnostic, 01-01, if you are
unsure.)

---

## Why this matters

Here is the most expensive sentence in applied machine learning:

> *"The model says customers who get the email spend 20 EUR more, so let's email everyone."*

It is expensive because the first half is true and the second half does not follow, and nothing
in the model's output warns you. Held-out accuracy will not catch it. Cross-validation will not
catch it. The model is doing its job correctly; the sentence is asking it a question it never
answered.

In this chapter we build that exact situation, watch a good model produce a 20 EUR figure whose
true value is 5, act on it, and count the missing money. Then we fix it two different ways, and
see why only one of the fixes is reliable.

This is also the interview question that separates people who have shipped something from people
who have finished a course.

## What you will be able to do

By the end of this chapter you can:

1. **Distinguish** three questions a model can be asked - what will happen, what is associated
   with what, and what happens if I intervene - and say which ones a fitted model answers.
2. **Define** a confounder and show one distorting an estimate, by hand and in code.
3. **Demonstrate** that a variable can be an excellent predictor and a useless lever.
4. **Compare** the two standard fixes - adjusting for the confounder and randomising the
   treatment - and say precisely when each fails.
5. **Recognise** the causal question hiding inside a request phrased as a prediction problem.

## Warm-up: retrieve, do not reread

From memory:

1. What are the two ingredients machine learning needs that ordinary programming does not?
2. Name the test that decides regression versus classification.
3. Why does a clustering have no held-out score?
4. In 00-02, why did the written policy beat the model even though the model scored 1.000?
5. What is drift?

<br>

*Answers: (1) data together with recorded outcomes - the answers. (2) is 7 nearer to 8 than to
2? (3) nothing says what the groups should have been, so there is no error to measure. (4) the
model could at best tie with a rule we already had, while adding data, training and maintenance
costs. (5) the world moving away from the data the model was fitted to, so a once-correct model
quietly stops being correct.*

## The situation

Tomás has been running the café's loyalty programme for a year, and marketing sends a monthly
offer email. Not to everybody: the system sends it to customers who look engaged - people who
visit often, who have used the app, who opened the last one.

At the end of the year he pulls the numbers and finds:

> Customers who received the email spent **20 EUR more** than customers who did not.

Marketing proposes the obvious next step: send it to everyone, and collect roughly 20 EUR per
customer.

**The question this chapter answers:** is that 20 EUR real? And if not, what would you have had
to do differently to find out?

## Six customers, and two different answers

Before any code, do this by hand. Six customers, three of whom got the email.

| Customer | Loyal? | Emailed? | Spend (EUR) |
|---|---|---|---|
| A | yes | yes | 90 |
| B | yes | yes | 100 |
| C | yes | no | 85 |
| D | no | yes | 50 |
| E | no | no | 40 |
| F | no | no | 45 |

**Step 1 - compare emailed with not emailed, the way Tomás did.**

- Emailed (A, B, D): `(90 + 100 + 50) / 3 = 240 / 3 = 80.00 EUR`
- Not emailed (C, E, F): `(85 + 40 + 45) / 3 = 170 / 3 = 56.67 EUR`
- Difference: **23.33 EUR**

**Step 2 - now compare only within the loyal customers, and only within the non-loyal ones.**

- Loyal, emailed (A, B): `(90 + 100) / 2 = 95.00`; loyal, not emailed (C): `85.00`. Difference:
  **10.00**
- Not loyal, emailed (D): `50.00`; not loyal, not emailed (E, F): `(40 + 45) / 2 = 42.50`.
  Difference: **7.50**
- Average of the two within-group differences: `(10.00 + 7.50) / 2 = 8.75 EUR`

**23.33 against 8.75.** Same six customers, same arithmetic, nothing hidden. The first number is
larger because the emailed group is stuffed with loyal customers, who spend more anyway. Compare
like with like and most of the gap disappears.

That gap has a name. **Loyalty is a confounder**: it influences who gets the email *and* it
influences spend, so a plain comparison of emailed against not-emailed mixes the effect of the
email with the effect of being the sort of person who gets emails.

### Predict before running

We are about to build 2,000 customers where **we set the true effect of the email to exactly
5.00 EUR**. Before running:

1. What will a model that sees only `emailed` report as the effect?
2. What will a model that also sees `loyalty` report?
3. If Tomás emails everyone who was not emailed, how much extra will he actually collect per
   customer?

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(3)
n_customers = 2000

# SYNTHETIC. We choose the truth, so we can check who recovers it.
loyalty = rng.normal(0, 1, n_customers)                       # an engagement score, standardised
p_email = 1 / (1 + np.exp(-1.5 * loyalty))                    # loyal customers are far more likely to be emailed
emailed = rng.random(n_customers) < p_email
noise = rng.normal(0, 8, n_customers)

TRUE_EFFECT = 5.0
spend = 40 + 15 * loyalty + TRUE_EFFECT * emailed + noise     # <- the email is worth exactly 5.00 EUR

customers = pd.DataFrame({"loyalty": loyalty, "emailed": emailed.astype(int), "spend": spend})
print(f"emailed: {customers['emailed'].mean():.0%} of customers")
print(f"mean spend, emailed    : {customers.loc[customers.emailed == 1, 'spend'].mean():.2f} EUR")
print(f"mean spend, not emailed: {customers.loc[customers.emailed == 0, 'spend'].mean():.2f} EUR")

**52.94 against 32.52 - a gap of 20.42 EUR**, for an email whose true worth we set to 5.00.

This is not a bug, a bad model or a small sample. It is the correct answer to the question
"how do emailed customers differ from non-emailed ones?" - which is simply not the question
anybody meant to ask.

In [ ]:
from sklearn.linear_model import LinearRegression

naive = LinearRegression().fit(customers[["emailed"]], customers["spend"])
adjusted = LinearRegression().fit(customers[["emailed", "loyalty"]], customers["spend"])

print(f"true effect, by construction        : {TRUE_EFFECT:.2f} EUR")
print(f"model seeing only 'emailed'         : {naive.coef_[0]:.2f} EUR")
print(f"model also seeing 'loyalty'         : {adjusted.coef_[0]:.2f} EUR")

The naive model reports **20.42**. The model that also sees loyalty reports **5.39** - close to
the truth of 5.00, and the remaining wobble is ordinary sampling noise (chapter 03-03 tells you
how much to expect).

The second model did not use a cleverer algorithm. Both are plain linear regression. The
difference is entirely in **which column was available**, which is the point: this is a question
about your data, not about your method. No amount of gradient boosting fixes it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 4))
for value, colour, label in [(1, "#0072B2", "emailed"), (0, "#D55E00", "not emailed")]:
    m = customers["emailed"] == value
    ax.scatter(customers.loc[m, "loyalty"], customers.loc[m, "spend"],
               s=8, alpha=0.5, color=colour, label=label)
ax.set_xlabel("Loyalty score (standardised)")
ax.set_ylabel("Spend (EUR)")
ax.set_title("The two groups are not comparable: emailed customers sit to the right")
ax.legend()
plt.show()

The picture is the whole argument.

The blue points are shifted **to the right**: emailed customers have higher loyalty. At any
given point on the horizontal axis - comparing customers of *equal* loyalty - the blue cloud
sits only slightly above the orange one, and that small vertical gap is the real 5 EUR. The big
20 EUR gap comes from comparing the average height of a right-hand cloud with the average height
of a left-hand one.

**The general shape of the error:** the groups you are comparing differ in more than the thing
you are studying. Everything in this chapter is a way of handling that sentence.

## Three different questions

The confusion has a tidy structure once you see it. There are three questions people ask of
data, they sound similar, and only the first two are answered by fitting a model.

| | The question | Example | What it needs |
|---|---|---|---|
| **1. Prediction** | What will Y be, for this row? | Will this customer spend a lot next month? | Association is enough. Any correlated feature helps |
| **2. Description / explanation** | Which features did the model use, and how? | The model leans heavily on `emailed` | Association, plus care in wording |
| **3. Causation** | What happens to Y **if I set** X? | If I email everyone, what changes? | An intervention, or strong assumptions you must state |

A fitted model answers 1 natively and 2 with effort. **It does not answer 3**, and it does not
say so. The coefficient `20.42` is a perfectly good description of an association; it becomes a
false statement only when someone reads it as "if we email a customer, they will spend 20.42
more".

The word that marks the boundary is **if I set**. Prediction observes the world; causation
changes it. As soon as a request contains *increase*, *reduce*, *should we*, *what if we*, or
*is it worth doing*, you have been handed a causal question, whatever the ticket title says.

**The one-line test, worth memorising:** *would this feature still have that value if I
intervened?* Umbrella sales predict rain superbly. Ban umbrellas and it will still rain.

---

## Failure lab: emailing everyone

Marketing acts on the 20.42. There are 976 customers who have never received the email.

Their arithmetic: `976 x 20.42 EUR = 19,929 EUR` of extra spend.

Because this is synthetic data we can do something no real analyst can: actually run the
intervention on the *same* customers, with the *same* randomness, and see what really happens.

**Predict before running:** how far off is the 19,929?

In [ ]:
# The counterfactual: the same 2,000 people, same loyalty, same noise, but everyone emailed.
spend_if_all_emailed = 40 + 15 * loyalty + TRUE_EFFECT * 1 + noise

untouched = customers["emailed"] == 0
promised = int(untouched.sum()) * naive.coef_[0]
actual = (spend_if_all_emailed - spend)[untouched.to_numpy()].sum()

print(f"customers not yet emailed : {int(untouched.sum())}")
print(f"gain promised by the model: {promised:8,.0f} EUR")
print(f"gain actually realised    : {actual:8,.0f} EUR")
print(f"overstatement             : {promised / actual:.1f}x  "
      f"({promised - actual:,.0f} EUR that never existed)")

### Diagnosis

**19,929 EUR promised, 4,880 EUR delivered.** The campaign was oversold by a factor of 4.1, and
roughly 15,000 EUR of the business case was imaginary.

Note carefully what did *not* go wrong:

- The model was not overfitted. It would score identically on held-out customers.
- The data was not dirty, the sample was not small, the code had no bug.
- The coefficient 20.42 is *correct* as a description: emailed customers really do spend 20.42
  more on average.

What went wrong is that a number describing **who differs from whom** was read as a number
describing **what changes if we act**. The email was never doing the work. Loyalty was doing the
work, and the email was standing next to it.

There is a second, quieter loss here that people usually miss. Because the campaign "worked" -
spending among newly emailed customers does rise by 5 - nobody discovers the error. The forecast
is simply revised, the next campaign is built on the same reasoning, and the organisation keeps
a belief that is wrong by 4x. **Causal errors do not announce themselves by failing.**

### The two fixes, and when each one fails

**Fix 1: adjust for the confounder.** Put `loyalty` in the model and compare like with like.
That is what gave us 5.39.

It works, and it has one brutal requirement: **you must know which variables confound, and you
must have measured them well.** Loyalty was recorded here. In a real system, the thing that
drives both treatment and outcome is often something nobody logged - intent, need, how the sales
rep felt about the customer. If it is not in the table, no model can adjust for it, and there is
nothing in the output that indicates a variable is missing.

Exercise E7 shows the sharper version of this problem: measuring the confounder *badly* only
partly fixes the bias, and a noisy proxy can leave you nearly as wrong as no adjustment at all.

**Fix 2: randomise who gets the email.** Assign it by coin flip. Then the two groups are alike
in loyalty, in intent, and in everything you never thought to measure - not because you
controlled for them, but because chance ignored them equally.

In [ ]:
# The same world, except the email is now assigned by a coin flip.
rng_exp = np.random.default_rng(3)
loyalty_e = rng_exp.normal(0, 1, n_customers)
emailed_e = rng_exp.random(n_customers) < 0.5            # <- no longer depends on loyalty
spend_e = 40 + 15 * loyalty_e + TRUE_EFFECT * emailed_e + rng_exp.normal(0, 8, n_customers)

experiment = pd.DataFrame({"emailed": emailed_e.astype(int), "spend": spend_e})
randomised = LinearRegression().fit(experiment[["emailed"]], experiment["spend"])

print(f"true effect                            : {TRUE_EFFECT:.2f} EUR")
print(f"naive comparison, observational data   : {naive.coef_[0]:.2f} EUR")
print(f"naive comparison, randomised experiment: {randomised.coef_[0]:.2f} EUR")

**4.36 against a truth of 5.00** - using the *same naive model* that was 4x wrong a moment ago,
on data collected differently.

This is the point that surprises people, so it is worth stating flatly: **randomisation is not a
statistical technique. It is a property of how the data was collected**, and it does more for
your conclusion than any modelling choice you can make afterwards. The analysis got simpler, not
cleverer.

The 0.64 shortfall is sampling noise - randomisation makes the estimate *unbiased*, not exact.
With 2,000 customers there is real wobble around the true value, and reporting 4.36 without an
interval would be its own kind of overclaiming. Chapter 03-03 gives you the tool for that.

| Fix | Fails when | Cost |
|---|---|---|
| Adjust for confounders | A confounder is unmeasured, or measured with noise (E7), or you adjusted for the wrong thing | Free if the data exists - but you can never prove it was enough |
| Randomise (A/B test) | You cannot ethically or practically assign the treatment; effects take years; the population being randomised is not the one you will deploy on | Time, traffic, engineering, and someone must accept withholding the treatment from half the users |
| Neither is possible | - | Then say the effect is unknown. A range with a stated assumption beats a confident wrong number |

### But the naive model is still a good *predictor*

Do not conclude that the naive model is worthless. Score it as a predictor on held-out
customers.

In [ ]:
from sklearn.metrics import mean_absolute_error

train, test = customers.iloc[:1400], customers.iloc[1400:]
predictor = LinearRegression().fit(train[["emailed"]], train["spend"])

print(f"MAE using only 'emailed'  : {mean_absolute_error(test['spend'], predictor.predict(test[['emailed']])):.2f} EUR")
print(f"MAE of the mean baseline  : {mean_absolute_error(test['spend'], [train['spend'].mean()] * len(test)):.2f} EUR")

**12.00 EUR against a baseline of 14.63.** The naive model is a genuinely useful predictor - if
you want to guess what a customer will spend, knowing whether they were emailed helps, because
it tells you something about how loyal they are.

So here is the sentence to take out of this chapter:

> **A variable can be an excellent predictor and a useless lever.**

`emailed` predicts spend because it carries information about loyalty. Pull the lever - email
everyone - and the information disappears, because now everyone is emailed and the variable no
longer distinguishes anyone. The prediction was never wrong. It was answering a different
question.

This is the deepest reason 07-06 and 07-07 are careful about the word "importance": a feature
that a model relies on heavily is not thereby something worth changing.

## Common misconceptions

**"Correlation does not imply causation - I know that one."**
Almost everybody can recite it and almost everybody acts against it, because in practice the
causal claim arrives disguised as a business recommendation rather than as the word "causes".
The recognisable form is not *"X causes Y"*, it is *"so we should do more X"*.

**"With enough features, the model becomes causal."**
Adding features can reduce confounding *if* the right ones are added and measured well. It can
also make things worse: adjusting for a variable that sits on the path from the treatment to the
outcome removes the very effect you are trying to measure. Which variables to include is a
question about how the world works, and it cannot be answered by looking at the data. 12-07.

**"A big dataset fixes this."**
No. More rows shrink the *noise*, not the *bias*. With 20 million customers the naive estimate
converges beautifully to 20.42, which is still four times the truth. This is worth remembering
whenever scale is offered as an answer to a validity problem.

**"Feature importance tells us what to change."**
It tells you what the model leaned on to predict. A hospital model may find that being in
intensive care predicts death; discharging people from intensive care does not help them.

**"We ran an A/B test, so we know the effect."**
You know the effect *for the population you randomised, at that time, at that dose*. Test on
engaged users in December and you have not measured what happens to lapsed users in July.

**"We can't randomise, so we can't say anything."**
Also wrong, and it is the mirror-image error. There are serious methods for causal estimation
from observational data, and there is always the option of stating a range with the assumption
attached. The unacceptable move is reporting an association as though it were an effect. 12-07
maps the field honestly.

---

## Exercises

Solutions with reasoning: `solutions/00_orientation/00-04_prediction_vs_cause_solutions.ipynb`.

### Quick understanding

**E1 (define).** Define a confounder in one sentence, then say what makes loyalty one in this
chapter's data.

**E2 (explain).** State the three questions a model can be asked, and say which of them a fitted
model answers without further assumptions.

**E3 (explain).** Why does "a variable can be an excellent predictor and a useless lever" follow
from what happened when everyone was emailed?

### Hand calculation

**E4 (calculate).** Six customers, spend in EUR:

| Customer | Region | Got a coupon? | Spend |
|---|---|---|---|
| P | city | yes | 60 |
| Q | city | yes | 70 |
| R | city | no | 55 |
| S | rural | yes | 30 |
| T | rural | no | 20 |
| U | rural | no | 25 |

Compute the naive coupon "effect" (coupon minus no-coupon), then the effect within each region,
then the average of the two within-region effects. Which number would you report to a manager,
and what one sentence would you attach to it?

**E5 (calculate).** In the chapter's data the naive estimate is 20.42 and the truth is 5.00.
Marketing wants to email 3,500 new customers and the email costs 0.40 EUR each to send. Compute
the promised profit and the real profit. At what true effect per customer does the campaign
break even?

### Coding

**E6 (code).** Rerun the simulation with the selection strength set to 0.0, 0.5, 1.5 and 3.0
(the `1.5` in `p_email`). Report the naive and adjusted coefficients for each. What happens to
the naive estimate when selection strength is 0, and why is that the whole idea behind
randomisation?

**E7 (code + diagnose).** Suppose loyalty is only measured approximately - the recorded score is
the true loyalty plus noise. Add noise of standard deviation 0.0, 0.5, 1.0 and 2.0 to the
loyalty column, adjust using the *noisy* version, and report the coefficient each time. What
does the pattern say about the sentence "we controlled for it"?

### Interpretation

**E8 (interpret).** A team reports: *"customers who use our mobile app churn 60% less, so we
should push app adoption."* Name the confounder you would look for first, describe the
comparison you would want to see instead, and say what evidence would actually settle it.

### Debugging

**E9 (diagnose).** A hospital model predicts that patients given a particular painkiller are
more likely to die within 30 days. Give three candidate explanations, ordered from most to least
likely, and say what you would ask the clinicians first.

### Exam and interview reasoning

**E10 (defend).** *"Our model has an R-squared of 0.91, so we understand what drives sales."*
Reply in four sentences, distinguishing prediction from cause and naming one concrete way the
statement could be true and useless at the same time.

**E11 (design).** You are asked whether a new onboarding tutorial increases 30-day retention. You
cannot run an A/B test because the tutorial has already been shipped to everyone. Describe two
approaches you could take, and state the assumption each one rests on.

### Transfer to a different situation

**E12 (design).** For each request, say whether it is prediction, description or causation, and
what would be needed to answer it honestly.
(a) Which customers should we call this week?
(b) Does calling customers increase renewals?
(c) Which factors best predict renewal?
(d) If we raise the price by 5%, how many customers do we lose?
(e) Which of our customers are most similar to the ones who left?

### Explain it to someone non-technical

**E13 (explain).** In under 80 words, explain to Tomás why the 20 EUR figure was real and the
plan built on it was not. Use one everyday comparison and say where it stops being accurate.

### Optional challenge

**E14 (design + code).** Build a case where adjusting makes things **worse**. Generate data in
which the email causes higher `app_opens`, and `app_opens` in turn causes higher spend. Estimate
the email's effect twice: once adjusting for `app_opens` and once not. Which one recovers the
true total effect, and why does "control for everything you have" fail here?

In [ ]:
# Your workspace. Still in memory: customers, loyalty, noise, emailed, spend,
# TRUE_EFFECT, naive, adjusted, experiment, LinearRegression, mean_absolute_error.

## Mastery check

Without scrolling up, can you:

- [ ] Define a confounder and give an example? *(If not: "Six customers, two different answers".)*
- [ ] Name the three questions, and which a fitted model answers? *(If not: "Three different
      questions".)*
- [ ] Explain why the campaign delivered a quarter of what was promised? *(If not: "Failure lab".)*
- [ ] Say when adjusting fails and when randomising fails? *(If not: "The two fixes".)*
- [ ] Explain how a variable can be a great predictor and a useless lever? *(If not: "But the
      naive model is still a good predictor".)*
- [ ] Spot the causal question inside "should we do more X"? *(If not: "Common misconceptions".)*

## What should now feel instinctive

1. **"Is this a prediction question or a decision question?"** - asked of every request, before
   any modelling.
2. **"Would this feature still have that value if I intervened?"** - the umbrella test.
3. **"Why did these rows get the treatment and those rows not?"** - the fastest way to find a
   confounder is to ask how the treatment was assigned.
4. **"What is missing from this table that affects both?"** - and if the answer is *something*,
   the estimate needs a caveat, not a decimal place.
5. **"Could this have been randomised?"** - and if it could have been, why was it not?

## Flashcards

| Question | Answer |
|---|---|
| Confounder | Something that influences both who gets the treatment and the outcome, so a plain comparison mixes the two |
| Three questions of a model | What will happen (prediction), what is associated (description), what changes if I act (causation) |
| Which does a fitted model answer? | The first two. Never the third, without an intervention or a stated assumption |
| The umbrella test | Would this feature still have that value if I intervened? |
| The linguistic tell for a causal question | "should we", "increase", "reduce", "what if we", "is it worth" |
| Why did adjusting work here? | Loyalty was recorded, so like could be compared with like |
| Why is randomisation stronger? | It balances the variables you never measured or thought of |
| Does more data fix confounding? | No. More rows shrink noise, not bias |
| A great predictor that is a useless lever | `emailed` - it predicts spend because it carries loyalty, but emailing everyone destroys that information |
| What does an A/B test actually establish? | The effect for that population, at that time, at that dose |

## Next

**Module 02 · Data literacy and EDA**, starting with **02-01 · What is a row?**

Orientation is finished. You can now tell an ML problem from a query, a rule and an experiment;
name the family of learning a task belongs to; and spot a causal question dressed as a
prediction. Everything so far has used data that was handed to you, clean, in four columns.

Real data is not like that, and almost every expensive mistake in this course's remaining
chapters is a data misunderstanding that was visible in the first hour. Module 02 is where you
learn to look.

*(Module 01 is the optional Python and pandas bridge. If you are unsure whether you need it, take
the diagnostic in 01-01 first - it takes twenty minutes and will tell you.)*

New terms are in [GLOSSARY.md](../../GLOSSARY.md).